In [32]:
import pandas as pd

In [33]:
lasso = pd.read_csv(
    "../submissions/linear_submission.csv"
)

xgb = pd.read_csv(
    "../submissions/xgb_random_search_submission.csv"
)

In [ ]:
import numpy as np

# =====================================================================
# 1. 宣告最佳化黃金權重 (由 SciPy SLSQP 求解 5-Fold OOF 預測值得出)
# =====================================================================
W_LASSO = 0.50071
W_XGB = 0.49929

# =====================================================================
# 2. 執行對數空間幾何加權平均
# =====================================================================
lasso_log = np.log1p(lasso["SalePrice"])
xgb_log = np.log1p(xgb["SalePrice"])

ensemble = pd.DataFrame({
    "Id": lasso["Id"],
    "SalePrice": np.expm1((W_LASSO * lasso_log) + (W_XGB * xgb_log)),
})

ensemble.head()

,Id,SalePrice
0,1461,120026.717384
1,1462,163328.165222
2,1463,182893.412719
3,1464,199099.550980
4,1465,188652.602611


In [35]:
ensemble.to_csv(
    "../submissions/ensemble_submission.csv",
    index=False
)

print("Ensemble Submission 已建立")

Ensemble Submission 已建立


In [36]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.linear_model import Lasso
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import cross_val_predict
from xgboost import XGBRegressor

# =========================================================
# 第 1 步：讀取訓練資料（特徵與 Log 空間的 target）
# =========================================================
print("1. 載入訓練資料中...")
X_train_linear = pd.read_csv("../data/X_train.csv")  # 已做 RobustScaler 的線性特徵
X_train_tree = pd.read_csv("../data/X_train_tree.csv")  # 給樹狀模型用的原始特徵
y_train = pd.read_csv("../data/y_train.csv").squeeze()  # 已做 log1p 的房價真值

# =========================================================
# 第 2 步：自動計算 Out-Of-Fold (OOF) 預測值並存檔
# =========================================================
print("2. 正在計算 Lasso 5-Fold OOF 預測值...")
lasso_model = Lasso(alpha=0.0005, max_iter=50000, random_state=42)
oof_lasso = cross_val_predict(lasso_model, X_train_linear, y_train, cv=5)

print("3. 正在計算 XGBoost 5-Fold OOF 預測值 (約需10~20秒)...")
xgb_model = XGBRegressor(
    n_estimators=3500,
    learning_rate=0.015,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=2,
    gamma=0.01,
    reg_alpha=0.01,
    reg_lambda=1.5,
    random_state=42,
    n_jobs=-1,
)
oof_xgb = cross_val_predict(xgb_model, X_train_tree, y_train, cv=5)

# 把算好的 OOF 存下來，以後重跑這格就隨時能用！
pd.Series(oof_lasso, name="SalePrice").to_csv(
    "lasso_oof_train.csv", index=False
)
pd.Series(oof_xgb, name="SalePrice").to_csv("xgb_oof_train.csv", index=False)
print("-> OOF 預測檔 lasso_oof_train.csv 與 xgb_oof_train.csv 已成功建立！\n")


# =========================================================
# 第 3 步：利用 SciPy 求解器尋找「最低 RMSE 黃金權重」
# =========================================================
def rmse_func(weights):
  w_lasso, w_xgb = weights
  blend_pred = (w_lasso * oof_lasso) + (w_xgb * oof_xgb)
  return root_mean_squared_error(y_train, blend_pred)


# 限制條件：w_lasso + w_xgb = 1，且權重介於 0 ~ 1 之間
constraints = {"type": "eq", "fun": lambda w: 1 - sum(w)}
bounds = [(0, 1), (0, 1)]

res = minimize(
    rmse_func,
    [0.5, 0.5],
    method="SLSQP",
    bounds=bounds,
    constraints=constraints,
)
best_w_lasso, best_w_xgb = res.x

print("=" * 40)
print("【優化求解完成！】")
print(f"Lasso 最優權重: {best_w_lasso:.5f}")
print(f"XGB   最優權重: {best_w_xgb:.5f}")
print(f"混合後本機 CV OOF RMSE: {res.fun:.5f}")
print("=" * 40 + "\n")

# =========================================================
# 第 4 步：讀取測試集預測檔，套用黃金權重產出最終 Submission
# =========================================================
print("4. 正在套用黃金權重建立最終 Submission...")
lasso_sub = pd.read_csv("../submissions/linear_submission.csv")
xgb_sub = pd.read_csv("../submissions/xgb_random_search_submission.csv")

# 記得：submission 裡的數字是原始金額，必須先 log1p 轉回 Log 空間做加權
lasso_sub_log = np.log1p(lasso_sub["SalePrice"])
xgb_sub_log = np.log1p(xgb_sub["SalePrice"])

ensemble_log = (best_w_lasso * lasso_sub_log) + (best_w_xgb * xgb_sub_log)

ensemble_submission = pd.DataFrame({
    "Id": lasso_sub["Id"],
    "SalePrice": np.expm1(ensemble_log),  # expm1 轉回現實金額
})

ensemble_submission.to_csv(
    "../submissions/ensemble_optimized_submission.csv", index=False
)
print("成功！最高勝率檔案已存為 ../submissions/ensemble_optimized_submission.csv")

1. 載入訓練資料中...
2. 正在計算 Lasso 5-Fold OOF 預測值...
3. 正在計算 XGBoost 5-Fold OOF 預測值 (約需10~20秒)...
-> OOF 預測檔 lasso_oof_train.csv 與 xgb_oof_train.csv 已成功建立！

【優化求解完成！】
Lasso 最優權重: 0.50071
XGB   最優權重: 0.49929
混合後本機 CV OOF RMSE: 0.10771

4. 正在套用黃金權重建立最終 Submission...
成功！最高勝率檔案已存為 ../submissions/ensemble_optimized_submission.csv
